# Machine Learning Analysis

This notebook applies multiple machine learning models to predict divorce status using demographic and socio-economic variables.

The objective is to evaluate different models and compare their ability to correctly identify divorced individuals. Special attention is given to recall due to class imbalance.

In [2]:
import pandas as pd

df = pd.read_csv("data/usa_00001.csv")

df['divorced'] = df['MARST'].isin([3, 4]).astype(int)
df['young'] = (df['AGE'] <= 25).astype(int)

df_sample = df.sample(n=10000, random_state=42)

print(df.shape)
print(df_sample.shape)

df_sample.head()

(3405809, 22)
(10000, 22)


,YEAR,SAMPLE,SERIAL,CBSERIAL,HHWT,CLUSTER,STATEICP,STRATA,GQ,PERNUM,...,AGE,MARST,EDUC,EDUCD,EMPSTAT,EMPSTATD,INCWAGE,POVERTY,divorced,young
2744464,2023,202301,1233692,2023000236556,62.0,2023012336921,54,210147,1,2,...,31,1,11,114,3,30,0,428,0,0
875515,2023,202301,382725,2023001353383,23.0,2023003827251,43,310112,1,3,...,5,6,1,12,0,0,999999,186,0,1
824648,2023,202301,360770,2023000988765,42.0,2023003607701,43,861812,1,1,...,41,1,7,71,1,10,100000,501,0,0
2806864,2023,202301,1262562,2023010051424,6.0,2023012625621,49,591148,4,1,...,18,6,6,63,3,30,0,0,0,1
1039079,2023,202301,454567,2023010080322,136.0,2023004545671,21,490017,3,1,...,64,6,8,81,3,30,0,0,0,0


## Sampling

A subset of the dataset is used to reduce computational cost while preserving overall data patterns.

In [4]:
df_sample['divorced'].value_counts()

divorced
0    8952
1    1048
Name: count, dtype: int64

## Feature Selection

The model uses AGE, SEX, EDUC, EMPSTAT, and young as predictors. These variables were selected based on their potential relationship with divorce observed during exploratory data analysis.

In [6]:
features = ['AGE', 'SEX', 'EDUC', 'EMPSTAT', 'young']

X = df_sample[features] 
y = df_sample['divorced']

## Train-Test Split

The dataset is divided into training and testing sets (80/20) to evaluate model performance on unseen data.

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y 
)

## Logistic Regression (Balanced)

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.5165
Precision: 0.17000871839581516
Recall: 0.9285714285714286
F1 Score: 0.28739867354458365
Confusion Matrix:
 [[838 952]
 [ 15 195]]


After applying class weights, recall improved significantly. This shows that handling class imbalance is critical for this problem.

## K-Nearest Neighbors

In [13]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

y_pred_knn = knn.predict(X_test_scaled)

print("KNN Accuracy:", accuracy_score(y_test, y_pred_knn))
print("KNN Precision:", precision_score(y_test, y_pred_knn, zero_division=0))
print("KNN Recall:", recall_score(y_test, y_pred_knn))
print("KNN F1 Score:", f1_score(y_test, y_pred_knn))

KNN Accuracy: 0.8835
KNN Precision: 0.21951219512195122
KNN Recall: 0.04285714285714286
KNN F1 Score: 0.07171314741035857


KNN achieved high accuracy but very low recall. This indicates that it is not effective for identifying divorced individuals in this dataset.

## Decision Tree

In [16]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=5, class_weight='balanced')
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

print("DT Accuracy:", accuracy_score(y_test, y_pred_dt))
print("DT Precision:", precision_score(y_test, y_pred_dt, zero_division=0))
print("DT Recall:", recall_score(y_test, y_pred_dt))
print("DT F1 Score:", f1_score(y_test, y_pred_dt))

DT Accuracy: 0.569
DT Precision: 0.17334669338677355
DT Recall: 0.8238095238095238
DT F1 Score: 0.28642384105960267


The decision tree improved recall substantially, making it better at identifying divorced individuals, although accuracy decreased.

## Random Forest

In [19]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    class_weight='balanced'
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("RF Accuracy:", accuracy_score(y_test, y_pred_rf))
print("RF Precision:", precision_score(y_test, y_pred_rf, zero_division=0))
print("RF Recall:", recall_score(y_test, y_pred_rf))
print("RF F1 Score:", f1_score(y_test, y_pred_rf))

RF Accuracy: 0.6055
RF Precision: 0.17725752508361203
RF Recall: 0.7571428571428571
RF F1 Score: 0.2872628726287263


Random Forest also performed well, but Logistic Regression achieved the highest recall. This means Random Forest is useful, but it was not the best model for the main objective of identifying divorced individuals.

In [21]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression (Balanced)",
        "KNN",
        "Decision Tree (Balanced)",
        "Random Forest (Balanced)"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_knn),
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_rf)
    ],
    "Precision": [
        precision_score(y_test, y_pred, zero_division=0),
        precision_score(y_test, y_pred_knn, zero_division=0),
        precision_score(y_test, y_pred_dt, zero_division=0),
        precision_score(y_test, y_pred_rf, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, y_pred),
        recall_score(y_test, y_pred_knn),
        recall_score(y_test, y_pred_dt),
        recall_score(y_test, y_pred_rf)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred),
        f1_score(y_test, y_pred_knn),
        f1_score(y_test, y_pred_dt),
        f1_score(y_test, y_pred_rf)
    ]
})

results = results.sort_values(by="Recall", ascending=False)
results

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression (Balanced),0.5165,0.170009,0.928571,0.287399
2,Decision Tree (Balanced),0.5690,0.173347,0.823810,0.286424
3,Random Forest (Balanced),0.6055,0.177258,0.757143,0.287263
1,KNN,0.8835,0.219512,0.042857,0.071713


## Model Comparison

The results show that:

- Logistic Regression achieves the highest recall.
- Decision Tree and Random Forest also perform well in recall.
- KNN has high accuracy but very low recall, so it is not useful for identifying divorced individuals.

F1 Score results also support this conclusion, since Logistic Regression keeps a reasonable balance between precision and recall.

## Final Conclusion

Among all models, Logistic Regression performed best because it achieved the highest recall.

Since the dataset is imbalanced, recall is more important than accuracy in this analysis. A model with high accuracy but low recall may fail to identify divorced individuals.

Overall, the results suggest that simpler linear models can outperform more complex models when the main objective is correctly identifying the minority class.

Future work could include hyperparameter tuning and testing more advanced models such as Gradient Boosting.